# Car models + TrajOpt compare

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/applications/car_trajopt.ipynb)

**Students on Colab:** open the badge → **File → Save a copy in Drive** → run from the top. The bootstrap cell clones `minilink` and installs `meshcat` on Colab.

This notebook has four parts:

1. **Part A — Modeling ladder** — state, inputs, minimal EoM `params`, and physics for each `jax_vehicles` rung.
2. **Part B — Open-loop compare** — same plants, constant inputs, forward-simulate XY / speed.
3. **Part C — Obstacle TrajOpt** — straight cruise with a soft sphere keepout (nine plants, including Engine).
4. **Part D — Corner TrajOpt** — 90° path + corridor (circuit SE bend, no obstacles) on the same nine plants.

`BicycleDynEngine` is included in every comparison below.

Runnable twin: [`demo_car_trajopt_compare.py`](../scripts/trajectory_optimization/demo_car_trajopt_compare.py).

---

## Recipe at a glance

| Step | Content |
| --- | --- |
| A | Ladder cards: $n$, $x$, $u$, params, $\dot x=f(x,u)$ |
| B | Open-loop: constant $u$, `compute_forced`, overlay paths |
| C | Obstacle mission: soft tracking + soft clearance |
| D | Corner mission: soft path distance + corridor hinge |


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone -b main https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


In [ ]:
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

from minilink.core.backends import configure_jax
from minilink.core.costs import QuadraticCost
from minilink.core.geometry import Sphere
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.vehicles.car_profile import (
    apply_car_profile,
    passenger_car_profile,
)
from minilink.dynamics.catalog.vehicles.jax_vehicles import (
    BicycleAcc,
    BicycleDyn,
    BicycleDynEngine,
    BicycleDynRate,
    BicycleDynServo,
    BicycleDynTauRate,
    BicycleKin,
    Holonomic,
    HolonomicAccel,
)
from minilink.planning.problems import PlanningProblem
from minilink.planning.spatial.collision import bind, car_outline, disc, point_probe
from minilink.planning.spatial.overlays import TrackCorridorOverlay
from minilink.planning.spatial.paths import from_waypoints
from minilink.planning.spatial.scene import Scene
from minilink.planning.spatial.shaping import (
    inverse_barrier,
    quadratic_excess,
    quadratic_hinge,
)
from minilink.planning.spatial.track import ReferenceTrack
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationPlanner,
)

configure_jax(enable_x64=True)


# Part A — Modeling ladder (`jax_vehicles`)

Planning and MPC often start with a cheap plant and climb fidelity until closed-loop behavior is credible. The Jax shelf lives in `minilink.dynamics.catalog.vehicles.jax_vehicles`: **module = JAX backend**, class names drop the `Jax` prefix. Default ports are stacked `u` with `y = x`.

**EoM `params` rule:** only independent quantities that enter $f$ (minimal set). Graphics axle offsets `a`/`b` are object attrs (and may be mirrored for skins); Dyn equations use a **centered CG** via `length` ($a=b=L/2$). Commanded inputs use the `_cmd` suffix.

Optional envelope for demos/MPC limits: `apply_car_profile(sys, passenger_car_profile())`.

| Class | $n$ | State gist | $u$ | EoM params | Physics |
| --- | ---: | --- | --- | --- | --- |
| `Holonomic` | 2 | $[x,y]$ | $[v_x,v_y]$ | `{}` | $\dot x = u$ |
| `HolonomicAccel` | 4 | $[x,y,v_x,v_y]$ | $[a_x,a_y]$ | `{}` | double integrator |
| `BicycleKin` | 3 | $[x,y,\theta]$ | $[v,\delta]$ | `length` | nonholonomic bicycle |
| `BicycleAcc` | 5 | $[x,y,\theta,v,\delta]$ | $[a_x,\dot\delta]$ | `length` | no-slip + accel / steer-rate |
| `BicycleDyn` | 6 | pose + body vel | $[\omega_r,\delta]$ | length, mass, tires, … | planar body + linear tires |
| `BicycleDynRate` | 8 | $+\,\omega_r,\delta$ | $[\dot\omega_r,\dot\delta]$ | + `Jw_rear`, `bw_rear` | actuator rates |
| `BicycleDynTauRate` | 8 | $+\,\omega_r,\delta$ | $[\tau,\dot\delta]$ | same as Rate | direct torque + steer rate |
| `BicycleDynServo` | 9 | $+\,\omega_r,\delta,\tau$ | $[\tau_\mathrm{cmd},\delta_\mathrm{cmd}]$ | + lag taus / rate max | lag on $\tau$ and $\delta$ |
| `BicycleDynEngine` | 9 | $+\,\omega_r,\delta,P$ | $[P_\mathrm{cmd},\delta_\mathrm{cmd}]$ | + `engine_tau`, `tau_sat`, `bw_engine`, `tau_fric` | power lag; $\tau=\mathrm{clip}(P/\omega,\pm\tau_\mathrm{sat})$; engine brake |


In [ ]:
def show_rung(sys, equation: str):
    """Print n, state/input labels, params, and a sample f evaluation."""
    u0 = np.asarray(sys.inputs["u"].nominal_value, dtype=float).reshape(-1)
    if u0.size == 0:
        u0 = np.zeros(sys.inputs["u"].dim)
    x0 = np.asarray(sys.x0, dtype=float).copy()
    # Mild nonzero IC so tire/kinematic terms are visible where relevant.
    if sys.n >= 4 and "vx" in (sys.state.labels or []):
        x0[3] = max(x0[3], 5.0)
    elif sys.n >= 4 and "v" in (sys.state.labels or []):
        x0[3] = max(x0[3], 5.0)
    dx = np.asarray(sys.f(x0, u0, 0.0))
    print(f"{sys.name}: n={sys.n}")
    print(f"  x labels: {list(sys.state.labels)}")
    print(f"  u labels: {list(sys.inputs['u'].labels)}")
    print(f"  params:   {sorted(sys.params.keys())}")
    print(f"  physics:  {equation}")
    print(f"  f(x0,u0) shape {dx.shape}, ||f||={np.linalg.norm(dx):.4g}")
    print()


show_rung(Holonomic(), r"dx/dt = u")
show_rung(HolonomicAccel(), r"d/dt [p; v] = [v; a]")
show_rung(BicycleKin(), r"ẋ=v cosθ, ẏ=v sinθ, θ̇=v tanδ / L")
show_rung(BicycleAcc(), r"pose as Kin; v̇=a_x; δ̇=u_δ")
show_rung(BicycleDyn(), r"M dv + C v + d(tires)=0; dq = N v")
show_rung(BicycleDynRate(), r"Dyn body + [ω̇_r; δ̇] = u")
show_rung(BicycleDynTauRate(), r"Jw ω̇ = τ − τ_ground; δ̇ = u_δ")
show_rung(BicycleDynServo(), r"τ̇=(τ_cmd−τ)/τ_τ; δ̇=clip((δ_cmd−δ)/τ_s)")
show_rung(
    BicycleDynEngine(),
    r"Ṗ=(P_cmd−P)/τ_e; τ=clip(P/ω,±τ_sat); Jω̇=τ−τ_g−τ_brake; δ̇=clip((δ_cmd−δ)/τ_s)",
)


In [ ]:
# Optional: apply a shared vehicle envelope (writes minimal Jax params + port/graphics hints).
sys_profiled = BicycleDyn()
apply_car_profile(sys_profiled, passenger_car_profile())
print(
    "after passenger_car_profile:",
    {k: round(float(sys_profiled.params[k]), 4) for k in ("length", "mass", "Ca", "Ck")},
)

# Optional params override — EoM reads the dict you pass (JAX-traceable).
sys = BicycleDyn()
x = np.array([0.0, 0.0, 0.0, 10.0, 1.0, 0.1])
u = np.array([10.0 / sys.params["r_r"], 0.05])
dx0 = np.asarray(sys.f(x, u, 0.0))
params_hi = dict(sys.params)
params_hi["Ca"] = 3.0 * params_hi["Ca"]
dx1 = np.asarray(sys.f(x, u, 0.0, params=params_hi))
print("||f(default) - f(Ca×3)|| =", float(np.linalg.norm(dx0 - dx1)))


# Part B — Open-loop compare (constant inputs)

**Question:** before optimizing, how do the nine plants respond to a simple constant command?

Shared setup: start near cruise speed $v_0$, hold a fixed steer intent $\delta$, integrate with `compute_forced` (`rk4_fixedsteps`). Inputs differ by rung (velocity / accel / wheel rate / torque / power), so paths will not match — the point is to see which fidelity adds lag, slip, and propulsion dynamics.


In [ ]:
@dataclass(frozen=True)
class ModelCase:
    key: str
    title: str
    color: str


PATH_COLORS = (
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
    "tab:purple",
    "tab:brown",
    "tab:pink",
    "tab:gray",
    "tab:olive",
)

MODEL_CASES = (
    ModelCase("holonomic", "Holonomic", PATH_COLORS[0]),
    ModelCase("kinematic", "BicycleKin", PATH_COLORS[1]),
    ModelCase("holonomic_dyn", "HolonomicAccel", PATH_COLORS[2]),
    ModelCase("bicycle_acc", "BicycleAcc", PATH_COLORS[3]),
    ModelCase("dynamic", "BicycleDyn", PATH_COLORS[4]),
    ModelCase("dynamic_rate", "BicycleDynRate", PATH_COLORS[5]),
    ModelCase("dynamic_taurate", "BicycleDynTauRate", PATH_COLORS[6]),
    ModelCase("dynamic_servo", "BicycleDynServo", PATH_COLORS[7]),
    ModelCase("dynamic_engine", "BicycleDynEngine", PATH_COLORS[8]),
)

OL_TF = 3.0
OL_DT = 0.05
OL_V0 = 8.0
OL_DELTA = 0.12
OL_TAU = 200.0
OL_P_CMD = 5000.0


def _speed_from_traj(case: ModelCase, traj: Trajectory) -> np.ndarray:
    if case.key == "holonomic":
        return np.hypot(traj.u[0, :], traj.u[1, :])
    if case.key == "holonomic_dyn":
        return np.hypot(traj.x[2, :], traj.x[3, :])
    if case.key == "kinematic":
        return traj.u[0, :]
    if case.key == "bicycle_acc":
        return traj.x[3, :]
    return np.hypot(traj.x[3, :], traj.x[4, :])


def openloop_setup(case: ModelCase):
    """Return (sys, x0, u_const) for a gentle left-turn cruise."""
    v0, delta = OL_V0, OL_DELTA
    if case.key == "holonomic":
        sys = Holonomic()
        return sys, np.zeros(2), np.array([v0, v0 * np.tan(delta)])
    if case.key == "holonomic_dyn":
        sys = HolonomicAccel()
        return sys, np.array([0.0, 0.0, v0, 0.0]), np.array([0.0, 0.5])
    if case.key == "kinematic":
        sys = BicycleKin()
        return sys, np.zeros(3), np.array([v0, delta])
    if case.key == "bicycle_acc":
        sys = BicycleAcc()
        return sys, np.array([0.0, 0.0, 0.0, v0, delta]), np.array([0.0, 0.0])
    if case.key == "dynamic":
        sys = BicycleDyn()
        r_r = float(sys.params["r_r"])
        return sys, np.array([0.0, 0.0, 0.0, v0, 0.0, 0.0]), np.array([v0 / r_r, delta])
    if case.key == "dynamic_rate":
        sys = BicycleDynRate()
        r_r = float(sys.params["r_r"])
        x0 = np.array([0.0, 0.0, 0.0, v0, 0.0, 0.0, v0 / r_r, delta])
        return sys, x0, np.array([0.0, 0.0])
    if case.key == "dynamic_taurate":
        sys = BicycleDynTauRate()
        r_r = float(sys.params["r_r"])
        x0 = np.array([0.0, 0.0, 0.0, v0, 0.0, 0.0, v0 / r_r, delta])
        return sys, x0, np.array([OL_TAU, 0.0])
    if case.key == "dynamic_servo":
        sys = BicycleDynServo()
        r_r = float(sys.params["r_r"])
        x0 = np.array([0.0, 0.0, 0.0, v0, 0.0, 0.0, v0 / r_r, 0.0, 0.0])
        return sys, x0, np.array([OL_TAU, delta])
    if case.key == "dynamic_engine":
        sys = BicycleDynEngine()
        r_r = float(sys.params["r_r"])
        x0 = np.array([0.0, 0.0, 0.0, v0, 0.0, 0.0, v0 / r_r, 0.0, 0.0])
        return sys, x0, np.array([OL_P_CMD, delta])
    raise KeyError(case.key)


@dataclass
class OpenLoopRun:
    case: ModelCase
    sys: object
    traj: Trajectory
    u: np.ndarray
    wall_s: float


openloop_runs: list[OpenLoopRun] = []
print(f"Open-loop compare — tf={OL_TF}s, dt={OL_DT}, v0={OL_V0}, delta={OL_DELTA}")
print(f"{'model':<18} {'u':>22} {'x_f':>8} {'y_f':>8} {'sim_s':>8}")
for case in MODEL_CASES:
    sys, x0, u = openloop_setup(case)
    sys.x0 = x0
    t0 = time.perf_counter()
    traj = sys.compute_forced(
        u=u,
        tf=OL_TF,
        dt=OL_DT,
        solver="rk4_fixedsteps",
        show=False,
        verbose=False,
    )
    wall_s = time.perf_counter() - t0
    openloop_runs.append(OpenLoopRun(case=case, sys=sys, traj=traj, u=u, wall_s=wall_s))
    u_str = "[" + ", ".join(f"{ui:.3g}" for ui in u) + "]"
    print(
        f"{case.title:<18} {u_str:>22} {traj.x[0, -1]:>8.2f} {traj.x[1, -1]:>8.2f} {wall_s:>8.3f}"
    )


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12.0, 10.0), sharex=True, sharey=True)
flat = axes.ravel()
for ax, run in zip(flat, openloop_runs):
    ax.plot(run.traj.x[0, :], run.traj.x[1, :], color=run.case.color, linewidth=1.8)
    ax.set_title(run.case.title)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
for ax in flat[len(openloop_runs) :]:
    ax.set_visible(False)
fig.suptitle("Open-loop paths (constant u)")
fig.tight_layout()
plt.show()

fig, (ax_path, ax_speed) = plt.subplots(1, 2, figsize=(12.0, 5.0))
for run in openloop_runs:
    ax_path.plot(
        run.traj.x[0, :],
        run.traj.x[1, :],
        color=run.case.color,
        linewidth=1.8,
        label=run.case.title,
    )
    ax_speed.plot(
        run.traj.t,
        _speed_from_traj(run.case, run.traj),
        color=run.case.color,
        linewidth=1.8,
        label=run.case.title,
    )
ax_path.set_title("All models — XY path")
ax_path.set_aspect("equal", adjustable="box")
ax_path.grid(True, alpha=0.25)
ax_path.legend(loc="best", fontsize=8)
ax_speed.set_xlabel("t [s]")
ax_speed.set_ylabel("speed [m/s]")
ax_speed.set_title("Longitudinal speed")
ax_speed.grid(True, alpha=0.25)
ax_speed.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()


# Part C — Obstacle scene, nine TrajOpt plants

**Question:** for one soft-goal / soft-clearance mission, how do paths, speeds, and solve times change as the plant climbs the ladder?

## C1. Mission parameters

Fixed horizon $t_f$ and collocation grid. Soft goal: reach world-$x \approx u_\mathrm{target}\, t_f$ near $y=0$, dodging one sphere keepout. Shared knobs below; each builder maps them onto that plant's $x$ / $u$ bounds.


In [ ]:
TF = 4.0
N_STEPS = 20
U_0 = 8.0
U_TARGET = 10.0
Y_START = 0.0
Y_GOAL = 0.0
HEADING_TARGET = 0.0
DELTA_MAX = 0.25
V_MAX = 20.0
SPEED_DOT_MAX = 3.0
STEERING_DOT_MAX = 2.0
W_REAR_DOT_MAX = 80.0
DELTA_DOT_MAX = 2.0
TAU_REAR_MAX = 5000.0
P_CMD_MAX = 100000.0

OBSTACLE_CENTER = (12.0, 0.2)
OBSTACLE_RADIUS = 0.4
OBSTACLE_MARGIN = 0.2
OBSTACLE_REPULSION_WEIGHT = 1000.0
OBSTACLE_REPULSION_EPS = 1.0
PLOT_BOUNDS = ((-2.0, U_TARGET * TF + 2.0), (-3.5, 2.0))



def terminal_x() -> float:
    return U_TARGET * TF


## C2. Workspace scene

Radius = geometric radius + planning margin. Soft clearance uses `inverse_barrier` on signed clearance — **no** hard collision constraint.


In [ ]:
keepout_radius = OBSTACLE_RADIUS + OBSTACLE_MARGIN
scene = Scene(obstacles=[Sphere(OBSTACLE_CENTER, keepout_radius)])
overlay = scene.as_visualizer(color="tab:red", opacity=0.45)

fig, ax = plt.subplots(figsize=(8.0, 3.5))
scene.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, show_density=False, title="Mission workspace")
ax.plot([0.0, terminal_x()], [Y_START, Y_GOAL], "k--", linewidth=1.0, label="soft straight goal")
ax.legend(loc="upper left")
plt.show()


## C3. Per-model builders

Each builder returns the plant plus TrajOpt ingredients: `x_start`, `x_ref`, `ubar`, `Q`, `R`, `S`.

Common theme: **no running weight on world-$x$** ($Q_{00}=0$); Y / heading / actuator penalties force weaving around the obstacle.


In [ ]:
@dataclass
class SolveRun:
    case: ModelCase
    sys: object
    traj: Trajectory
    success: bool
    cost: float | None
    solve_s: float | None
    total_s: float


def build_holonomic():
    sys = Holonomic()
    sys.inputs["u"].lower_bound = np.array([0.0, -V_MAX])
    sys.inputs["u"].upper_bound = np.array([V_MAX, V_MAX])
    x_start = np.array([0.0, Y_START])
    x_ref = np.array([terminal_x(), Y_GOAL])
    ubar = np.array([U_TARGET, 0.0])
    Q = np.diag([0.0, 10.0])
    R = np.diag([0.5, 0.5])
    S = np.diag([1.0, 10.0])
    return sys, x_start, x_ref, ubar, Q, R, S


def build_holonomic_dyn():
    sys = HolonomicAccel()
    sys.state.lower_bound[2] = 0.0
    sys.state.upper_bound[2] = V_MAX
    sys.state.lower_bound[3] = -V_MAX
    sys.state.upper_bound[3] = V_MAX
    sys.inputs["u"].lower_bound = np.array([-SPEED_DOT_MAX, -SPEED_DOT_MAX])
    sys.inputs["u"].upper_bound = np.array([SPEED_DOT_MAX, SPEED_DOT_MAX])
    x_start = np.array([0.0, Y_START, U_0, 0.0])
    x_ref = np.array([terminal_x(), Y_GOAL, U_TARGET, 0.0])
    ubar = np.array([0.0, 0.0])
    Q = np.diag([0.0, 10.0, 0.1, 0.1])
    R = np.diag([1.0, 1.0])
    S = np.diag([1.0, 10.0, 10.0, 1.0])
    return sys, x_start, x_ref, ubar, Q, R, S


def build_kinematic():
    sys = BicycleKin()
    sys.inputs["u"].lower_bound = np.array([0.0, -DELTA_MAX])
    sys.inputs["u"].upper_bound = np.array([V_MAX, DELTA_MAX])
    x_start = np.array([0.0, Y_START, 0.0])
    x_ref = np.array([terminal_x(), Y_GOAL, HEADING_TARGET])
    ubar = np.array([U_TARGET, 0.0])
    Q = np.diag([0.0, 10.0, 1.0])
    R = np.diag([0.5, 10.0])
    S = np.diag([1.0, 10.0, 100.0])
    return sys, x_start, x_ref, ubar, Q, R, S


def build_bicycle_acc():
    sys = BicycleAcc()
    sys.state.lower_bound[3] = 0.0
    sys.state.upper_bound[3] = V_MAX
    sys.state.lower_bound[4] = -DELTA_MAX
    sys.state.upper_bound[4] = DELTA_MAX
    sys.inputs["u"].lower_bound = np.array([-SPEED_DOT_MAX, -STEERING_DOT_MAX])
    sys.inputs["u"].upper_bound = np.array([SPEED_DOT_MAX, STEERING_DOT_MAX])
    x_start = np.array([0.0, Y_START, 0.0, U_0, 0.0])
    x_ref = np.array([terminal_x(), Y_GOAL, HEADING_TARGET, U_TARGET, 0.0])
    ubar = np.array([0.0, 0.0])
    Q = np.diag([0.0, 10.0, 1.0, 0.1, 100.0])
    R = np.diag([1.0, 10.0])
    S = np.diag([1.0, 10.0, 100.0, 10.0, 100.0])
    return sys, x_start, x_ref, ubar, Q, R, S


def build_dynamic():
    sys = BicycleDyn()
    r_r = sys.params["r_r"]
    w_rear_max = 1.2 * U_TARGET / r_r
    sys.inputs["u"].lower_bound = np.array([0.0, -DELTA_MAX])
    sys.inputs["u"].upper_bound = np.array([w_rear_max, DELTA_MAX])
    x_start = np.array([0.0, Y_START, 0.0, U_0, 0.0, 0.0])
    x_ref = np.array([terminal_x(), Y_GOAL, HEADING_TARGET, U_TARGET, 0.0, 0.0])
    ubar = np.array([U_TARGET / r_r, 0.0])
    Q = np.diag([0.0, 10.0, 1.0, 0.1, 0.1, 0.1])
    R = np.diag([0.5, 10.0])
    S = np.diag([1.0, 10.0, 100.0, 10.0, 0.0, 0.1])
    return sys, x_start, x_ref, ubar, Q, R, S


def build_dynamic_rate():
    sys = BicycleDynRate()
    r_r = sys.params["r_r"]
    w_rear_max = 1.2 * U_TARGET / r_r
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = w_rear_max
    sys.state.lower_bound[7] = -DELTA_MAX
    sys.state.upper_bound[7] = DELTA_MAX
    sys.inputs["u"].lower_bound = np.array([-W_REAR_DOT_MAX, -DELTA_DOT_MAX])
    sys.inputs["u"].upper_bound = np.array([W_REAR_DOT_MAX, DELTA_DOT_MAX])
    x_start = np.array([0.0, Y_START, 0.0, U_0, 0.0, 0.0, U_0 / r_r, 0.0])
    x_ref = np.array(
        [terminal_x(), Y_GOAL, HEADING_TARGET, U_TARGET, 0.0, 0.0, U_TARGET / r_r, 0.0]
    )
    ubar = np.array([0.0, 0.0])
    Q = np.diag([0.0, 10.0, 1.0, 0.1, 0.1, 0.1, 0.1, 100.0])
    R = np.diag([1.0, 10.0])
    S = np.diag([1.0, 10.0, 100.0, 10.0, 0.0, 0.1, 0.1, 100.0])
    return sys, x_start, x_ref, ubar, Q, R, S


def build_dynamic_taurate():
    sys = BicycleDynTauRate()
    r_r = sys.params["r_r"]
    w_rear_max = 1.2 * U_TARGET / r_r
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = w_rear_max
    sys.state.lower_bound[7] = -DELTA_MAX
    sys.state.upper_bound[7] = DELTA_MAX
    sys.inputs["u"].lower_bound = np.array([-TAU_REAR_MAX, -DELTA_DOT_MAX])
    sys.inputs["u"].upper_bound = np.array([TAU_REAR_MAX, DELTA_DOT_MAX])
    x_start = np.array([0.0, Y_START, 0.0, U_0, 0.0, 0.0, U_0 / r_r, 0.0])
    x_ref = np.array(
        [terminal_x(), Y_GOAL, HEADING_TARGET, U_TARGET, 0.0, 0.0, U_TARGET / r_r, 0.0]
    )
    ubar = np.array([0.0, 0.0])
    Q = np.diag([0.0, 10.0, 1.0, 0.1, 0.1, 0.1, 0.1, 100.0])
    R = np.diag([1e-4, 10.0])
    S = np.diag([1.0, 10.0, 100.0, 10.0, 0.0, 0.1, 0.1, 100.0])
    return sys, x_start, x_ref, ubar, Q, R, S


def build_dynamic_servo():
    sys = BicycleDynServo()
    r_r = sys.params["r_r"]
    w_rear_max = 1.2 * U_TARGET / r_r
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = w_rear_max
    sys.state.lower_bound[7] = -DELTA_MAX
    sys.state.upper_bound[7] = DELTA_MAX
    sys.inputs["u"].lower_bound = np.array([-TAU_REAR_MAX, -DELTA_MAX])
    sys.inputs["u"].upper_bound = np.array([TAU_REAR_MAX, DELTA_MAX])
    x_start = np.array([0.0, Y_START, 0.0, U_0, 0.0, 0.0, U_0 / r_r, 0.0, 0.0])
    x_ref = np.array(
        [
            terminal_x(),
            Y_GOAL,
            HEADING_TARGET,
            U_TARGET,
            0.0,
            0.0,
            U_TARGET / r_r,
            0.0,
            0.0,
        ]
    )
    ubar = np.array([0.0, 0.0])
    Q = np.diag([0.0, 10.0, 1.0, 0.1, 0.1, 0.1, 0.1, 100.0, 0.0])
    R = np.diag([1e-4, 10.0])
    S = np.diag([1.0, 10.0, 100.0, 10.0, 0.0, 0.1, 0.1, 100.0, 0.0])
    return sys, x_start, x_ref, ubar, Q, R, S



def build_dynamic_engine():
    sys = BicycleDynEngine()
    r_r = sys.params["r_r"]
    w_rear_max = 1.2 * U_TARGET / r_r
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = w_rear_max
    sys.state.lower_bound[7] = -DELTA_MAX
    sys.state.upper_bound[7] = DELTA_MAX
    sys.state.lower_bound[8] = -P_CMD_MAX
    sys.state.upper_bound[8] = P_CMD_MAX
    sys.inputs["u"].lower_bound = np.array([-P_CMD_MAX, -DELTA_MAX])
    sys.inputs["u"].upper_bound = np.array([P_CMD_MAX, DELTA_MAX])
    x_start = np.array([0.0, Y_START, 0.0, U_0, 0.0, 0.0, U_0 / r_r, 0.0, 0.0])
    x_ref = np.array(
        [
            terminal_x(),
            Y_GOAL,
            HEADING_TARGET,
            U_TARGET,
            0.0,
            0.0,
            U_TARGET / r_r,
            0.0,
            0.0,
        ]
    )
    ubar = np.array([0.0, 0.0])
    Q = np.diag([0.0, 10.0, 1.0, 0.1, 0.1, 0.1, 0.1, 100.0, 0.0])
    R = np.diag([1e-10, 10.0])
    S = np.diag([1.0, 10.0, 100.0, 10.0, 0.0, 0.1, 0.1, 100.0, 0.0])
    return sys, x_start, x_ref, ubar, Q, R, S


BUILDERS = {
    "holonomic": build_holonomic,
    "holonomic_dyn": build_holonomic_dyn,
    "kinematic": build_kinematic,
    "bicycle_acc": build_bicycle_acc,
    "dynamic": build_dynamic,
    "dynamic_rate": build_dynamic_rate,
    "dynamic_taurate": build_dynamic_taurate,
    "dynamic_servo": build_dynamic_servo,
    "dynamic_engine": build_dynamic_engine,
}


## C4. Cost, transcription, and solve

For each plant: quadratic tracking + soft obstacle clearance → `PlanningProblem` → direct collocation (JAX).


In [ ]:
def speed_profile(case: ModelCase, traj: Trajectory) -> np.ndarray:
    if case.key == "holonomic":
        return np.hypot(traj.u[0, :], traj.u[1, :])
    if case.key == "holonomic_dyn":
        return np.hypot(traj.x[2, :], traj.x[3, :])
    if case.key == "kinematic":
        return traj.u[0, :]
    if case.key == "bicycle_acc":
        return traj.x[3, :]
    return np.hypot(traj.x[3, :], traj.x[4, :])


def solve_case(case: ModelCase, scene: Scene) -> SolveRun:
    sys, x_start, x_ref, ubar, Q, R, S = BUILDERS[case.key]()
    tracking_cost = QuadraticCost.from_system(sys, Q=Q, R=R, S=S, xbar=x_ref, ubar=ubar)
    obstacle_cost = scene.clearance_field(bind(sys, point_probe())).as_cost(
        weight=OBSTACLE_REPULSION_WEIGHT,
        shaping=inverse_barrier(epsilon=OBSTACLE_REPULSION_EPS),
    )
    problem = PlanningProblem(
        sys=sys, tf=TF, x_start=x_start, cost=tracking_cost + obstacle_cost
    )
    planner = TrajectoryOptimizationPlanner(
        problem,
        n_steps=N_STEPS,
        transcription="direct_collocation",
        compile_backend="jax",
        record_solve_time=True,
        optimizer_options={"maxiter": 500, "ftol": 1e-1},
    )
    t0 = time.perf_counter()
    traj = planner.solve().trajectory
    total_s = time.perf_counter() - t0
    result = planner.last_optimization_result
    return SolveRun(
        case=case,
        sys=sys,
        traj=traj,
        success=bool(result.success),
        cost=None if result.cost is None else float(result.cost),
        solve_s=None if result.solve_time_s is None else float(result.solve_time_s),
        total_s=total_s,
    )


runs = [solve_case(case, scene) for case in MODEL_CASES]


## C5. Compare results

Summary table, $2\times 4$ path panel, then overlay of all XY paths and reconstructed speeds.


In [ ]:
print("TrajOpt obstacle scene — model comparison")
print(f"  tf={TF}s, n_steps={N_STEPS}, u_target={U_TARGET} m/s")
print(f"  obstacle={OBSTACLE_CENTER}, keepout R={keepout_radius}")
print(f"{'model':<18} {'success':>8} {'solve_s':>9} {'total_s':>9} {'J*':>12} {'x_f':>8}")
for run in runs:
    solve_s = "—" if run.solve_s is None else f"{run.solve_s:.3f}"
    cost = "—" if run.cost is None else f"{run.cost:.1f}"
    print(
        f"{run.case.title:<18} {str(run.success):>8} {solve_s:>9} "
        f"{run.total_s:>9.3f} {cost:>12} {run.traj.x[0, -1]:>8.1f}"
    )


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12.0, 10.0), sharex=True, sharey=True)
flat = axes.ravel()
for ax, run in zip(flat, runs):
    scene.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, show_density=False, title="")
    ax.plot(
        run.traj.x[0, :],
        run.traj.x[1, :],
        color=run.case.color,
        linewidth=1.8,
        label="planned",
    )
    solve_s = "—" if run.solve_s is None else f"{run.solve_s:.2f}s"
    ax.set_title(f"{run.case.title}  (solve {solve_s})")
    ax.legend(loc="upper left", fontsize=8)
for ax in flat[len(runs) :]:
    ax.set_visible(False)
fig.suptitle("Planned paths (scene obstacle overlay)")
fig.tight_layout()
plt.show()


In [ ]:
fig, (ax_path, ax_speed) = plt.subplots(1, 2, figsize=(12.0, 5.0))

scene.plot(show=False, ax=ax_path, bounds=PLOT_BOUNDS, show_density=False, title="")
for run in runs:
    ax_path.plot(
        run.traj.x[0, :],
        run.traj.x[1, :],
        color=run.case.color,
        linewidth=1.8,
        label=run.case.title,
    )
ax_path.set_title("All models — XY path")
ax_path.legend(loc="upper left")

for run in runs:
    ax_speed.plot(
        run.traj.t,
        speed_profile(run.case, run.traj),
        color=run.case.color,
        linewidth=1.8,
        label=run.case.title,
    )
ax_speed.set_xlabel("t [s]")
ax_speed.set_ylabel("speed [m/s]")
ax_speed.set_title("Longitudinal speed")
ax_speed.grid(True, alpha=0.25)
ax_speed.legend(loc="best", fontsize=8)

fig.tight_layout()
plt.show()


## C6. Animate obstacle (optional)

Set `ANIMATE_OBSTACLE = True` to play all nine with the same obstacle overlay. Skip in headless runs.


In [ ]:
ANIMATE_OBSTACLE = False  # set True to play interactive animations

if ANIMATE_OBSTACLE:
    for run in runs:
        print(f"Animating obstacle / {run.case.title}...")
        run.sys.traj = run.traj
        run.sys.animate(
            run.traj, overlays=[overlay], renderer="meshcat", html=False, native=False
        )
else:
    print("Skipped obstacle animations (set ANIMATE_OBSTACLE = True to play).")


# Part D — 90° corner (path + corridor)

**Question:** same nine plants, but now the geometry is a reference track: soft path distance + corridor hinge, **no obstacles**.

Centerline is the SE bend of the wide technical circuit (`demo_mpc_circuit`): approach eastbound at $y=-10$, then turn north along $x=17$. Place the car just before the bend and plan a short open-loop horizon.

Bicycle plants use JAX + `car_outline` (same collision body as the MPC demos). Holonomic plants use NumPy + `disc` — path-distance fields under JAX are unreliable for the point plants.

## C1. Track and knobs


## D1. Track and knobs


In [ ]:
# Circuit SE corner (east → north), with short lead-in/out
CORNER_XY = np.array(
    [
        [5.0, -10.0],
        [10.5, -10.0],
        [13.7101, -9.6985],
        [14.5, -9.3301],
        [15.2139, -8.8302],
        [15.8302, -8.2139],
        [16.3301, -7.5],
        [16.6985, -6.7101],
        [16.9240, -5.8682],
        [17.0, -5.0],
        [17.0, -2.0],
        [17.0, 2.0],
        [17.0, 5.0],
    ]
)
CORNER_HALF_WIDTH = 2.5
CORNER_S0 = 5.0
CORNER_TF = 3.0
CORNER_N_STEPS = 15
CORNER_U_0 = 6.0
CORNER_U_TARGET = 8.0
CORNER_DELTA_MAX = 0.55
CORNER_V_MAX = 20.0
CORNER_PATH_WEIGHT = 40.0
CORNER_CORRIDOR_WEIGHT = 25.0
CORNER_CAR_LENGTH = 2.4
CORNER_CAR_WIDTH = 0.2
CORNER_CAR_MARGIN = 0.05

track = ReferenceTrack(from_waypoints(CORNER_XY), half_width=CORNER_HALF_WIDTH)


def pose_on_track(s: float):
    xy = np.asarray(track.path.sample(s), dtype=float)
    tangent = track.path.tangent(s)
    theta = float(np.arctan2(tangent[1], tangent[0]))
    if abs(np.cos(2.0 * theta)) > 1.0 - 1e-9:
        theta += 1e-4
    return xy, theta


xy0, th0 = pose_on_track(CORNER_S0)
xyf, thf = pose_on_track(track.path.total_length - 0.5)
corner_pad = CORNER_HALF_WIDTH + 1.0
CORNER_PLOT_BOUNDS = (
    (float(CORNER_XY[:, 0].min()) - corner_pad, float(CORNER_XY[:, 0].max()) + corner_pad),
    (float(CORNER_XY[:, 1].min()) - corner_pad, float(CORNER_XY[:, 1].max()) + corner_pad),
)

fig, ax = track.plot(show=False, bounds=CORNER_PLOT_BOUNDS, title="")
ax.scatter([xy0[0]], [xy0[1]], color="tab:green", s=40, zorder=5, label="start")
ax.scatter([xyf[0]], [xyf[1]], color="tab:red", s=40, zorder=5, label="soft exit")
ax.legend(loc="best")
ax.set_title(f"Corner track (len={track.path.total_length:.1f} m)")
ax.set_aspect("equal", adjustable="box")
plt.show()
print(f"start={xy0}, theta={th0:.3f}  |  exit={xyf}, theta={thf:.3f}")


## D2. Corner builders

Cruise regulation in $Q$ (path/corridor handle $x,y$). Holonomic plants add a soft terminal at the exit so they finish the turn.


In [ ]:
def corner_build_holonomic():
    sys = Holonomic()
    sys.inputs["u"].lower_bound = np.array([-CORNER_V_MAX, -CORNER_V_MAX])
    sys.inputs["u"].upper_bound = np.array([CORNER_V_MAX, CORNER_V_MAX])
    x_start = np.array([xy0[0], xy0[1]])
    x_ref = np.array([xyf[0], xyf[1]])
    ubar = np.array([CORNER_U_TARGET, 0.0])
    Q = np.diag([0.0, 0.0])
    R = np.diag([0.5, 0.5])
    S = np.diag([30.0, 30.0])
    return sys, x_start, x_ref, ubar, Q, R, S, "numpy", disc(0.3)


def corner_build_holonomic_dyn():
    sys = HolonomicAccel()
    sys.state.lower_bound[2] = -CORNER_V_MAX
    sys.state.upper_bound[2] = CORNER_V_MAX
    sys.state.lower_bound[3] = -CORNER_V_MAX
    sys.state.upper_bound[3] = CORNER_V_MAX
    sys.inputs["u"].lower_bound = np.array([-3.0, -3.0])
    sys.inputs["u"].upper_bound = np.array([3.0, 3.0])
    x_start = np.array(
        [xy0[0], xy0[1], CORNER_U_0 * np.cos(th0), CORNER_U_0 * np.sin(th0)]
    )
    x_ref = np.array(
        [xyf[0], xyf[1], CORNER_U_TARGET * np.cos(thf), CORNER_U_TARGET * np.sin(thf)]
    )
    ubar = np.zeros(2)
    Q = np.diag([0.0, 0.0, 0.15, 0.15])
    R = np.diag([1.0, 1.0])
    S = np.diag([30.0, 30.0, 1.0, 1.0])
    return sys, x_start, x_ref, ubar, Q, R, S, "numpy", disc(0.3)


def corner_build_kinematic():
    sys = BicycleKin()
    sys.inputs["u"].lower_bound = np.array([0.0, -CORNER_DELTA_MAX])
    sys.inputs["u"].upper_bound = np.array([CORNER_V_MAX, CORNER_DELTA_MAX])
    x_start = np.array([xy0[0], xy0[1], th0])
    x_ref = np.zeros(3)
    ubar = np.array([CORNER_U_TARGET, 0.0])
    Q = np.diag([0.0, 0.0, 0.0])
    R = np.diag([0.5, 10.0])
    S = np.diag([0.0, 0.0, 0.0])
    return sys, x_start, x_ref, ubar, Q, R, S, "jax", car_outline(2.4, 0.2, margin=0.05)


def corner_build_bicycle_acc():
    sys = BicycleAcc()
    sys.state.lower_bound[3] = 0.0
    sys.state.upper_bound[3] = CORNER_V_MAX
    sys.state.lower_bound[4] = -CORNER_DELTA_MAX
    sys.state.upper_bound[4] = CORNER_DELTA_MAX
    sys.inputs["u"].lower_bound = np.array([-3.0, -2.5])
    sys.inputs["u"].upper_bound = np.array([3.0, 2.5])
    x_start = np.array([xy0[0], xy0[1], th0, CORNER_U_0, 0.0])
    x_ref = np.array([0.0, 0.0, 0.0, CORNER_U_TARGET, 0.0])
    ubar = np.zeros(2)
    Q = np.diag([0.0, 0.0, 0.0, 0.15, 40.0])
    R = np.diag([1.0, 10.0])
    S = np.diag([0.0, 0.0, 0.0, 0.15, 40.0])
    return sys, x_start, x_ref, ubar, Q, R, S, "jax", car_outline(2.4, 0.2, margin=0.05)


def corner_build_dynamic():
    sys = BicycleDyn()
    r_r = sys.params["r_r"]
    sys.inputs["u"].lower_bound = np.array([0.0, -CORNER_DELTA_MAX])
    sys.inputs["u"].upper_bound = np.array([90.0, CORNER_DELTA_MAX])
    x_start = np.array([xy0[0], xy0[1], th0, CORNER_U_0, 0.0, 0.0])
    x_ref = np.array([0.0, 0.0, 0.0, CORNER_U_TARGET, 0.0, 0.0])
    ubar = np.array([CORNER_U_TARGET / r_r, 0.0])
    Q = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0])
    R = np.diag([0.5, 22.0])
    S = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0])
    return sys, x_start, x_ref, ubar, Q, R, S, "jax", car_outline(2.4, 0.2, margin=0.05)


def corner_build_dynamic_rate():
    sys = BicycleDynRate()
    r_r = sys.params["r_r"]
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = 90.0
    sys.state.lower_bound[7] = -CORNER_DELTA_MAX
    sys.state.upper_bound[7] = CORNER_DELTA_MAX
    sys.inputs["u"].lower_bound = np.array([-80.0, -2.0])
    sys.inputs["u"].upper_bound = np.array([80.0, 2.0])
    x_start = np.array(
        [xy0[0], xy0[1], th0, CORNER_U_0, 0.0, 0.0, CORNER_U_0 / r_r, 0.0]
    )
    x_ref = np.array(
        [0.0, 0.0, 0.0, CORNER_U_TARGET, 0.0, 0.0, CORNER_U_TARGET / r_r, 0.0]
    )
    ubar = np.zeros(2)
    Q = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0])
    R = np.diag([1.0, 22.0])
    S = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0])
    return sys, x_start, x_ref, ubar, Q, R, S, "jax", car_outline(2.4, 0.2, margin=0.05)


def corner_build_dynamic_taurate():
    sys = BicycleDynTauRate()
    r_r = sys.params["r_r"]
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = 90.0
    sys.state.lower_bound[7] = -CORNER_DELTA_MAX
    sys.state.upper_bound[7] = CORNER_DELTA_MAX
    sys.inputs["u"].lower_bound = np.array([-5000.0, -2.0])
    sys.inputs["u"].upper_bound = np.array([5000.0, 2.0])
    x_start = np.array(
        [xy0[0], xy0[1], th0, CORNER_U_0, 0.0, 0.0, CORNER_U_0 / r_r, 0.0]
    )
    x_ref = np.array(
        [0.0, 0.0, 0.0, CORNER_U_TARGET, 0.0, 0.0, CORNER_U_TARGET / r_r, 0.0]
    )
    ubar = np.zeros(2)
    Q = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0])
    R = np.diag([1e-4, 22.0])
    S = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0])
    return sys, x_start, x_ref, ubar, Q, R, S, "jax", car_outline(2.4, 0.2, margin=0.05)


def corner_build_dynamic_servo():
    sys = BicycleDynServo()
    r_r = sys.params["r_r"]
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = 90.0
    sys.state.lower_bound[7] = -CORNER_DELTA_MAX
    sys.state.upper_bound[7] = CORNER_DELTA_MAX
    sys.inputs["u"].lower_bound = np.array([-5000.0, -CORNER_DELTA_MAX])
    sys.inputs["u"].upper_bound = np.array([5000.0, CORNER_DELTA_MAX])
    x_start = np.array(
        [xy0[0], xy0[1], th0, CORNER_U_0, 0.0, 0.0, CORNER_U_0 / r_r, 0.0, 0.0]
    )
    x_ref = np.array(
        [0.0, 0.0, 0.0, CORNER_U_TARGET, 0.0, 0.0, CORNER_U_TARGET / r_r, 0.0, 0.0]
    )
    ubar = np.zeros(2)
    Q = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0, 0.0])
    R = np.diag([1e-4, 22.0])
    S = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0, 0.0])
    return sys, x_start, x_ref, ubar, Q, R, S, "jax", car_outline(2.4, 0.2, margin=0.05)



def corner_build_dynamic_engine():
    sys = BicycleDynEngine()
    r_r = sys.params["r_r"]
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = 90.0
    sys.state.lower_bound[7] = -CORNER_DELTA_MAX
    sys.state.upper_bound[7] = CORNER_DELTA_MAX
    sys.state.lower_bound[8] = -P_CMD_MAX
    sys.state.upper_bound[8] = P_CMD_MAX
    sys.inputs["u"].lower_bound = np.array([-P_CMD_MAX, -CORNER_DELTA_MAX])
    sys.inputs["u"].upper_bound = np.array([P_CMD_MAX, CORNER_DELTA_MAX])
    x_start = np.array(
        [xy0[0], xy0[1], th0, CORNER_U_0, 0.0, 0.0, CORNER_U_0 / r_r, 0.0, 0.0]
    )
    x_ref = np.array(
        [0.0, 0.0, 0.0, CORNER_U_TARGET, 0.0, 0.0, CORNER_U_TARGET / r_r, 0.0, 0.0]
    )
    ubar = np.zeros(2)
    Q = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0, 0.0])
    R = np.diag([1e-10, 22.0])
    S = np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0, 0.0])
    geom = car_outline(CORNER_CAR_LENGTH, CORNER_CAR_WIDTH, margin=CORNER_CAR_MARGIN)
    return sys, x_start, x_ref, ubar, Q, R, S, "jax", geom


CORNER_BUILDERS = {
    "holonomic": corner_build_holonomic,
    "holonomic_dyn": corner_build_holonomic_dyn,
    "kinematic": corner_build_kinematic,
    "bicycle_acc": corner_build_bicycle_acc,
    "dynamic": corner_build_dynamic,
    "dynamic_rate": corner_build_dynamic_rate,
    "dynamic_taurate": corner_build_dynamic_taurate,
    "dynamic_servo": corner_build_dynamic_servo,
    "dynamic_engine": corner_build_dynamic_engine,
}


## D3. Solve (path + corridor)

Cost: cruise quadratic + `track.distance_field` + `track.corridor_field`.


In [ ]:
def solve_corner_case(case: ModelCase) -> SolveRun:
    sys, x_start, x_ref, ubar, Q, R, S, backend, geom = CORNER_BUILDERS[case.key]()
    body = bind(sys, geom)
    tracking_cost = QuadraticCost.from_system(sys, Q=Q, R=R, S=S, xbar=x_ref, ubar=ubar)
    path_cost = track.distance_field(body).as_cost(
        weight=CORNER_PATH_WEIGHT, shaping=quadratic_excess(threshold=0.1)
    )
    corridor_cost = track.corridor_field(body).as_cost(
        weight=CORNER_CORRIDOR_WEIGHT, shaping=quadratic_hinge(threshold=0.0)
    )
    problem = PlanningProblem(
        sys=sys,
        tf=CORNER_TF,
        x_start=x_start,
        cost=tracking_cost + path_cost + corridor_cost,
    )
    planner = TrajectoryOptimizationPlanner(
        problem,
        n_steps=CORNER_N_STEPS,
        transcription="direct_collocation",
        compile_backend=backend,
        record_solve_time=True,
        optimizer_method="scipy_slsqp",
        optimizer_options={"maxiter": 150, "ftol": 0.1},
    )
    t0 = time.perf_counter()
    traj = planner.solve().trajectory
    total_s = time.perf_counter() - t0
    result = planner.last_optimization_result
    return SolveRun(
        case=case,
        sys=sys,
        traj=traj,
        success=bool(result.success),
        cost=None if result.cost is None else float(result.cost),
        solve_s=None if result.solve_time_s is None else float(result.solve_time_s),
        total_s=total_s,
    )


corner_runs = [solve_corner_case(case) for case in MODEL_CASES]

print("TrajOpt 90° corner — path + corridor")
print(
    f"  tf={CORNER_TF}s, n_steps={CORNER_N_STEPS}, u_target={CORNER_U_TARGET} m/s, "
    f"half_width={CORNER_HALF_WIDTH} m"
)
print(f"{'model':<18} {'success':>8} {'solve_s':>9} {'total_s':>9} {'J*':>12}")
for run in corner_runs:
    solve_s = "—" if run.solve_s is None else f"{run.solve_s:.3f}"
    cost = "—" if run.cost is None else f"{run.cost:.1f}"
    print(
        f"{run.case.title:<18} {str(run.success):>8} {solve_s:>9} "
        f"{run.total_s:>9.3f} {cost:>12}"
    )


## D4. Compare paths and speeds


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12.0, 10.0), sharex=True, sharey=True)
flat = axes.ravel()
for ax, run in zip(flat, corner_runs):
    track.plot(show=False, ax=ax, bounds=CORNER_PLOT_BOUNDS, title="")
    ax.plot(
        run.traj.x[0, :],
        run.traj.x[1, :],
        color=run.case.color,
        linewidth=1.8,
        label="planned",
    )
    solve_s = "—" if run.solve_s is None else f"{run.solve_s:.2f}s"
    ax.set_title(f"{run.case.title}  (solve {solve_s})")
    ax.legend(loc="upper left", fontsize=8)
    ax.set_aspect("equal", adjustable="box")
for ax in flat[len(corner_runs) :]:
    ax.set_visible(False)
fig.suptitle("Corner mission — planned paths")
fig.tight_layout()
plt.show()


In [ ]:
fig, (ax_path, ax_speed) = plt.subplots(1, 2, figsize=(12.0, 5.0))
track.plot(show=False, ax=ax_path, bounds=CORNER_PLOT_BOUNDS, title="")
for run in corner_runs:
    ax_path.plot(
        run.traj.x[0, :],
        run.traj.x[1, :],
        color=run.case.color,
        linewidth=1.8,
        label=run.case.title,
    )
ax_path.set_title("All models — XY path")
ax_path.legend(loc="best", fontsize=8)
ax_path.set_aspect("equal", adjustable="box")

for run in corner_runs:
    ax_speed.plot(
        run.traj.t,
        speed_profile(run.case, run.traj),
        color=run.case.color,
        linewidth=1.8,
        label=run.case.title,
    )
ax_speed.set_xlabel("t [s]")
ax_speed.set_ylabel("speed [m/s]")
ax_speed.set_title("Longitudinal speed")
ax_speed.grid(True, alpha=0.25)
ax_speed.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()


## D5. Animate corner (optional)


In [ ]:
ANIMATE_CORNER = False  # set True to play interactive animations

if ANIMATE_CORNER:
    corridor_overlay = TrackCorridorOverlay(track)
    for run in corner_runs:
        print(f"Animating corner / {run.case.title}...")
        run.sys.traj = run.traj
        run.sys.camera_scale = 18.0
        run.sys.animate(
            run.traj,
            overlays=[corridor_overlay],
            renderer="meshcat",
            html=False,
            native=False,
        )
else:
    print("Skipped corner animations (set ANIMATE_CORNER = True to play).")
